In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from scipy.stats import norm, uniform
import arviz

from aspire import Aspire
from aspire.samples import Samples
from aspire.utils import configure_logger
from aspire.plot import plot_comparison

configure_logger("INFO", additional_loggers=["aspire_ptemcee"])

# autoreload for development
%load_ext autoreload
%autoreload 2

In [ ]:
outdir = Path("outdir") / "unimodal_to_bimodal"
outdir.mkdir(parents=True, exist_ok=True)

figure_outdir = Path("figures")
figure_outdir.mkdir(parents=True, exist_ok=True)

table_outdir = Path("evidence_tables")
table_outdir.mkdir(parents=True, exist_ok=True)

seed = 42
rng = np.random.default_rng(seed)

In [ ]:
# Define bimodal likelihood in 4 dimensions
dims = 4


def log_likelihood(samples):
    x = samples.x
    # First mode: centered at (0, 0, 0, 0) with std of 1
    mode1 = norm.logpdf(x, loc=0, scale=1).sum(axis=-1)
    # Second mode: centered at (5, 5, 5, 5) with std of 1
    mode2 = norm.logpdf(x, loc=5, scale=1).sum(axis=-1)
    # Combine the two modes
    return np.logaddexp(mode1, mode2) - np.log(2)  # Normalize by the number of modes


def log_prior(samples):
    x = samples.x
    # Uniform prior over a large range
    return uniform.logpdf(x, loc=-10, scale=20).sum(axis=-1)


# Analytic ln-evidence
ln_z_analytic = -dims * np.log(20)
analytic_samples = np.concatenate(
    [
        rng.normal(loc=0, scale=1, size=(5000, dims)),
        rng.normal(loc=5, scale=1, size=(5000, dims)),
    ],
    axis=0,
)
analytic_samples = Samples(x=analytic_samples)

n_initial_samples = 5000

# Initial samples from a unimodal distribution (centered at (0, 0, 0, 0))
initial_samples = rng.normal(loc=0, scale=1, size=(n_initial_samples, dims))
initial_samples = Samples(x=initial_samples)

# Initial samples that are a mix of the prior and one mode of the likelihood

fraction = 0.2
n_prior_samples = int(fraction * n_initial_samples)
n_likelihood_samples = n_initial_samples - n_prior_samples

initial_samples_mixed = np.concatenate(
    [
        rng.normal(loc=0, scale=1, size=(n_likelihood_samples, dims)),
        rng.uniform(low=-10, high=10, size=(n_prior_samples, dims)),
    ],
    axis=0,
)
rng.shuffle(initial_samples_mixed)
initial_samples_mixed = Samples(x=initial_samples_mixed)

prior_bounds = {p: (-10, 10) for p in initial_samples.parameters}

In [ ]:
aspire = Aspire(
    log_likelihood=log_likelihood,
    log_prior=log_prior,
    dims=4,
    prior_bounds=prior_bounds,
    seed=seed,
    dtype="float64",
)

aspire.fit(initial_samples)

n_final_samples = 10_000
n_steps = 200

samples, smc_history = aspire.sample_posterior(
    1000,
    sampler="smc",
    return_history=True,
    target_efficiency=0.9,
    rng=rng,
    sampler_kwargs=dict(
        n_steps=n_steps,
    ),
    n_final_samples=n_final_samples,
)
smc_likelihood_evals_total = aspire.n_likelihood_evaluations
smc_likelihood_evals = smc_likelihood_evals_total - int(n_steps * n_final_samples)

In [ ]:
samples_ptmcmc, ptcmcm_history = aspire.sample_posterior(
    n_samples=None,
    sampler="ptemcee",
    return_history=True,
    ntemps=5,
    nwalkers=250,
    nsteps=2000,
    rng=rng,
)
ptmcmc_likelihood_evals = aspire.n_likelihood_evaluations

In [ ]:
mean_autocor = np.mean(samples_ptmcmc.autocorrelation_time[0])
ptmcmc_burn_in = int(2 * mean_autocor)
ptmcmc_burn_in_fraction = ptmcmc_burn_in / samples_ptmcmc.chain.shape[1]
print(f"Mean autocorrelation time: {mean_autocor:.2f} steps")
samples_ptmcmc_pp = samples_ptmcmc.post_process(
    burn_in=ptmcmc_burn_in,
    thin=int(mean_autocor),
)
ptmcmc_posterior_samples = samples_ptmcmc_pp.cold_chain()

In [ ]:
aspire_mixed = Aspire(
    log_likelihood=log_likelihood,
    log_prior=log_prior,
    dims=4,
    prior_bounds=prior_bounds,
    seed=seed,
    dtype="float64",
)
aspire_mixed.fit(initial_samples_mixed)

n_steps = 200
n_final_samples = 10_000

samples_mixed, smc_history_mixed = aspire_mixed.sample_posterior(
    1000,
    sampler="smc",
    return_history=True,
    target_efficiency=0.95,
    rng=rng,
    sampler_kwargs=dict(
        n_steps=n_steps,
    ),
    n_final_samples=n_final_samples,
)
smc_mixed_likelihood_evals_total = aspire_mixed.n_likelihood_evaluations
smc_mixed_likelihood_evals = smc_mixed_likelihood_evals_total - int(
    n_steps * n_final_samples
)

In [ ]:
samples_ptmcmc_mixed, ptcmcm_history = aspire_mixed.sample_posterior(
    n_samples=None,
    sampler="ptemcee",
    return_history=True,
    ntemps=5,
    nwalkers=250,
    nsteps=2000,
    rng=rng,
)
ptmcmc_mixed_likelihood_evals = aspire_mixed.n_likelihood_evaluations

In [ ]:
mean_autocor = np.mean(samples_ptmcmc_mixed.autocorrelation_time[0])
ptmcmc_mixed_burn_in = int(2 * mean_autocor)
ptmcmc_mixed_burn_in_fraction = (
    ptmcmc_mixed_burn_in / samples_ptmcmc_mixed.chain.shape[1]
)
print(f"Mean autocorrelation time: {mean_autocor:.2f} steps")
samples_ptmcmc_mixed_pp = samples_ptmcmc_mixed.post_process(
    burn_in=ptmcmc_mixed_burn_in,
    thin=int(mean_autocor),
)
ptmcmc_posterior_samples_mixed = samples_ptmcmc_mixed_pp.cold_chain()

In [ ]:
parameter_labels = [f"$\\theta_{i}$" for i in range(dims)]
labels = [
    "Analytic",
    "ASPIRE (unimodal init.)",
    "ASPIRE (mixed init.)",
    "PTMCMC",
]
lw = 2

rc_params = {
    "legend.fontsize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "axes.linewidth": 1.5,
    "xtick.major.size": 4,
    "xtick.major.width": 1,
    "xtick.minor.size": 2,
    "xtick.minor.width": 1,
    "ytick.major.size": 4,
    "ytick.major.width": 1,
    "ytick.minor.size": 2,
    "ytick.minor.width": 1,
    "grid.linewidth": 1.0,
    "axes.grid": True,
}

with plt.rc_context(rc=rc_params, fname="plots.style"):
    fig = plot_comparison(
        analytic_samples,
        samples,
        samples_mixed,
        ptmcmc_posterior_samples,
        labels=labels,
        plot_density=False,
        plot_datapoints=False,
        levels=[0.68, 0.95],
        per_samples_kwargs=[
            dict(
                color="black",
                contour_kwargs=dict(linestyles="-", linewidths=lw),
                hist_kwargs=dict(
                    color="black", histtype="step", linestyle="-", density=True, lw=lw
                ),
            ),
            dict(
                color="C1",
                contour_kwargs=dict(linestyles="--", linewidths=lw),
                hist_kwargs=dict(
                    color="C1", histtype="step", linestyle="--", density=True, lw=lw
                ),
            ),
            dict(
                color="darkred",
                contour_kwargs=dict(linestyles="-.", linewidths=lw),
                hist_kwargs=dict(
                    color="darkred",
                    histtype="step",
                    linestyle="-.",
                    density=True,
                    lw=lw,
                ),
            ),
            dict(
                color="C0",
                contour_kwargs=dict(linestyles=":", linewidths=lw),
                hist_kwargs=dict(
                    color="C0", histtype="step", linestyle=":", density=True, lw=lw
                ),
                labels=parameter_labels,
            ),
        ],
    )
    fig.savefig(figure_outdir / "unimodal_to_multimodal_comparison.pdf")
    plt.show()

## Evidence table

In [ ]:
def fmt(x, ndp=2):
    if x is None or x == "":
        return ""
    return f"{x:.{ndp}f}"


def fmt_int(x):
    if x is None or x == "":
        return ""
    return f"{int(x):,}"


def fmt_ratio(n, d, ndp=2):
    if n in (None, "") or d in (None, "", 0):
        return ""
    return f"{n / d:.{ndp}f}"


def within_3sigma(val, sigma, truth):
    return abs(val - truth) <= (3 * sigma)


def format_logz(val, sigma, truth):
    zero_centered_val = val - truth
    s = fmt(zero_centered_val)
    if within_3sigma(val, sigma, truth):
        return rf"\textbf{{{s}}}"
    return s


ptmcmc_logz, ptmcmc_logz_err = samples_ptmcmc_pp.log_evidence_stepping_stone()
ptmcmc_mixed_logz, ptmcmc_mixed_logz_err = (
    samples_ptmcmc_mixed_pp.log_evidence_stepping_stone()
)

rows = [
    # (
    #     "Analytic",
    #     "",
    #     "",
    #     fmt(ln_z_analytic),
    #     "",
    # ),
    (
        "ASPIRE SMC (unimodal init.)",
        fmt_int(len(samples)),
        f"{fmt_int(smc_likelihood_evals)} / {fmt_int(smc_likelihood_evals_total)}",
        format_logz(samples.log_evidence, samples.log_evidence_error, ln_z_analytic),
        fmt(samples.log_evidence_error),
    ),
    (
        "ASPIRE SMC (mixed init.)",
        fmt_int(len(samples_mixed)),
        f"{fmt_int(smc_mixed_likelihood_evals)} / {fmt_int(smc_mixed_likelihood_evals_total)}",
        format_logz(
            samples_mixed.log_evidence, samples_mixed.log_evidence_error, ln_z_analytic
        ),
        fmt(samples_mixed.log_evidence_error),
    ),
    (
        "PTMCMC (unimodal init.)",
        fmt_int(len(ptmcmc_posterior_samples)),
        fmt_int(ptmcmc_likelihood_evals),
        format_logz(ptmcmc_logz, ptmcmc_logz_err, ln_z_analytic),
        fmt(ptmcmc_logz_err),
    ),
    (
        "PTMCMC (mixed init.)",
        fmt_int(len(ptmcmc_posterior_samples_mixed)),
        fmt_int(ptmcmc_mixed_likelihood_evals),
        format_logz(ptmcmc_mixed_logz, ptmcmc_mixed_logz_err, ln_z_analytic),
        fmt(ptmcmc_mixed_logz_err),
    ),
]

latex_lines = [
    r"\begin{tabular}{p{5cm} C{2.5cm} C{3cm} c c}",
    r"\toprule",
    r"Method & Posterior samples & Likelihood evaluations & $\log Z - \log Z_{\mathrm{analytic}}$ & Uncertainty \\",
    r"\midrule",
]

for row in rows:
    latex_lines.append(" & ".join(row) + r" \\")

latex_lines.extend(
    [
        r"\bottomrule",
        r"\end{tabular}",
    ]
)

latex = "\n".join(latex_lines) + "\n"
print(latex)

tex_path = table_outdir / "evidence_table_unimodal_to_multimodal.tex"
tex_path.write_text(latex)

In [ ]:
# Compute rhat from MCMC chains
from aspire_analysis_tools.utils import compute_rhat

rhat_a = compute_rhat(samples_ptmcmc.cold_chain())
rhat_mixed = compute_rhat(samples_ptmcmc_mixed.cold_chain())

print(f"R-hat for PTMCMC (unimodal init.): {rhat_a}")
print(f"R-hat for PTMCMC (mixed init.): {rhat_mixed}")

for title, chain in zip(
    ["PTMCMC", "PTMCMC mixed"],
    [samples_ptmcmc, samples_ptmcmc_mixed],
):
    posterior_chain = chain.cold_chain()
    var_names = posterior_chain.parameters
    # Compute rhat for each temperature

    rhat_values = {}

    for i, beta in enumerate(chain.betas):
        print(f"Beta: {beta}")
        chain_at_temp = chain.at_temperature(i)

        arviz_data = arviz.from_dict(
            {"posterior": chain_at_temp.chain.transpose(1, 0, 2)}
        )
        rhat = arviz.rhat(arviz_data)
        rhat_values[beta] = dict(zip(var_names, rhat["posterior"].values))

    # Plot R-hat values for each parameter across temperatures
    plt.figure(figsize=(10, 6))
    for param in var_names:
        rhat_param = [rhat_values[beta][param] for beta in chain.betas]
        plt.plot(chain.betas, rhat_param, marker="o", label=param)
    plt.axhline(1.1, color="red", linestyle="--", label="R-hat = 1.1")
    plt.xscale("log")
    plt.xlabel("Inverse Temperature (beta)")
    plt.ylabel("R-hat")
    plt.title("R-hat Values Across Temperatures for " + title)
    plt.legend()
    plt.show()